# Évaluation et démo : test dans Colab

Avant de lancer : **Exécution > Modifier le type d'exécution > GPU T4**.

In [ ]:
!nvidia-smi -L

## 1. Dépôt et dépendances
Clone du dépôt GitHub et installation d'Ultralytics.

In [ ]:
%cd /content
!git clone -q https://github.com/oehtgit/projet_detection_objets_yolo11
%cd /content/projet_detection_objets_yolo11
!pip install -q ultralytics gdown
!ls eval demo

## 2. Données de la Personne 1 (zip 70/10/20, 2,2 Go, quelques minutes)
Les données restent sur le disque local de Colab (`/content/datasets/VOC`), comme l'attend `data/data.yaml`.

In [ ]:
!mkdir -p /content/datasets
!gdown -q 1JD_19qsvIUdjzPuSI2k6zhK-OIr2wMbs -O /content/datasets/VOC_pret.zip
!cd /content/datasets && unzip -q VOC_pret.zip && rm VOC_pret.zip
!ls /content/datasets/VOC/images/test | wc -l   # doit afficher 4300

## 3. Test de l'évaluation avec le modèle d'origine
On doit retrouver la baseline de la Personne 1 : **mAP50 ≈ 0,809 et mAP50-95 ≈ 0,626**. Si c'est le cas, le script est bon.

In [ ]:
!python eval/evaluate.py --device 0 --models avant=yolo11n.pt

In [ ]:
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv('eval/results/metrics_test.csv'))
display(Image('eval/results/figures/comparaison_globale.png', width=700))

## 4. Test de la démo sur quelques images du test

In [ ]:
!python demo/demo.py --source /content/datasets/VOC/images/test --max-frames 6 --device 0
import glob
for f in sorted(glob.glob('demo/outputs/test/*.jpg'))[:3]:
    display(Image(f, width=500))

## 5. Évaluation finale avant / après (avec le best.pt de la Personne 3)
Compare le modèle d'origine, le modèle fine-tuné et le fine-tuné à deux tailles d'image plus petites (précision et FPS sur le GPU T4).

In [ ]:
# best.pt est déjà dans le dépôt (train/weights/best.pt)
!python eval/evaluate.py --device 0 --models avant=yolo11n.pt apres=train/weights/best.pt \
    apres_416=train/weights/best.pt@416 apres_320=train/weights/best.pt@320

In [ ]:
display(pd.read_csv('eval/results/metrics_test.csv'))
for f in sorted(glob.glob('eval/results/figures/*.png')):
    display(Image(f, width=700))

## 6. Vidéos de démo côte à côte (pour la présentation)
Cette section n'a besoin que des cellules 1 et 2 (pas des données).

Renommer les vidéos en `repas.mp4` et `rue.mp4` sur le PC, puis lancer la cellule suivante et les choisir.

In [ ]:
%cd /content/projet_detection_objets_yolo11
from google.colab import files
up = files.upload()  # choisir repas.mp4 (et rue.mp4)
print(list(up))

In [ ]:
import os
for video in ['repas.mp4', 'rue.mp4']:
    if os.path.exists(video):
        !python demo/demo.py --source {video} --compare yolo11n.pt train/weights/best.pt --device 0
!ls -lh demo/outputs

Télécharger les vidéos résultats (modèle d'origine à gauche, fine-tuné à droite) :

In [ ]:
import glob
for f in glob.glob('demo/outputs/*_comparaison.mp4'):
    files.download(f)